# Step 1 — Setup checks

The first thing to run on a real GPU. It answers the three questions still open
about the scoring pipeline, so that everything built on top of it rests on
facts rather than assumptions:

1. **How many relation classes does the model have?** Three, or three plus a
   "no relation" class? This changes how we treat low-confidence pairs.
2. **Which way do the arrows point?** Every measurement about "paths to the
   conclusion" depends on getting the direction right.
3. **How slow is all-pairs vs neighbours-only?** This single choice dominates
   the whole training compute budget.

Run the cells top to bottom. Where a cell asks you to look at the output and
decide something, actually stop and look. Write what you find into
`docs/run_log.md` and, if it is new to you, into a note in `docs/learning/`.

## Setup

Makes the repo importable and installs what the scoring code needs. With the VS Code Colab extension and a synced folder, the clone is skipped because the repo is already here.

In [1]:
import os, sys, subprocess

REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"

if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)
subprocess.run(["git", "pull", "--quiet"], check=False)

sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "networkx", "pyyaml"], check=True)

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", commit or "(not a git checkout)")
print("ready")

commit: 468ddb6
ready


In [8]:
import os, sys, subprocess

# The repo is PRIVATE. Cloning it on Colab needs a read-only access token.
# Store it once in Colab Secrets (key icon, left sidebar) as GITHUB_TOKEN,
# with Notebook access on. It is read from there, never written into this
# notebook or the stored git remote. See the instructions above.

REPO_USER, REPO_NAME = "ookino", "rlvr-argument-mining"
CLEAN_URL = f"https://github.com/{REPO_USER}/{REPO_NAME}.git"

def _token():
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

if not os.path.exists("reward"):
    tok = _token()
    url = f"https://{tok}@github.com/{REPO_USER}/{REPO_NAME}.git" if tok else CLEAN_URL
    subprocess.run(["git", "clone", url, REPO_NAME], check=True)
    subprocess.run(["git", "-C", REPO_NAME, "remote", "set-url", "origin", CLEAN_URL], check=True)
    os.chdir(REPO_NAME)

sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "networkx", "pyyaml"], check=True)

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("commit:", commit or "(not a git checkout)")
print("ready")

commit: 468ddb6
ready


## Load the relation model

This is the model Ramon's group trained. First run downloads it (a few hundred MB), which is slow; later runs are instant. It goes on the GPU automatically if one is connected.

In [2]:
from reward.ari import ARI

ari = ARI()
print("running on:", ari.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/945 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.42GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

running on: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

## Question 1 — how many relation classes?

Our code assumes three: Inference (support), Conflict (contradict), Rephrase
(restate). A pair that scores below the confidence cutoff is treated as "no
relation".

If the model prints **three** labels, that assumption is correct. If it prints
**four**, there is also a built-in "no relation" class, and we should note how
often it wins rather than relying only on the cutoff.

In [3]:
print("relation classes the model knows:")
for idx, label in ari.id2label.items():
    print(f"  {idx}: {label}")

relation classes the model knows:
  0: No-Relation
  1: Inference
  2: Conflict
  3: Rephrase


## Question 2 — which way do the arrows point?

The raven argument is unambiguous: steps 0 and 1 both **support** step 2, the
conclusion. So correct arrows point INTO step 2.

Look at the output. If you see `step 0 --RA--> step 2` and `step 1 --RA--> step
2`, the direction matches what `features.py` assumes and the connectivity
measurement is correct. If the arrows point the other way (out of step 2), the
convention is reversed and we flip one line in `ari.py`. Either outcome is fine;
the point is to know before building on it.

In [4]:
from reward.xaif_build import build_trace

text = (
    "All ravens are black.\n"
    "Every bird in this garden is a raven.\n"
    "Therefore every bird in this garden is black.\n"
    "The answer is black."
)
trace = build_trace(text)

print("steps:")
for i, s in enumerate(trace.steps):
    mark = "  <- conclusion" if i == trace.conclusion_index else ""
    print(f"  {i}: {s}{mark}")

result = ari.identify(trace.steps, window=None)

print("\nrelations found:")
for r in result.relations:
    print(f"  step {r.source}  --{r.kind}-->  step {r.target}   (conf {r.confidence:.2f})")
print(f"\npairs scored: {result.n_pairs_scored}   below threshold: {result.n_below_threshold}")

steps:
  0: All ravens are black.
  1: Every bird in this garden is a raven.
  2: Therefore every bird in this garden is black.
  3: The answer is black.  <- conclusion

relations found:
  step 2  --RA-->  step 0   (conf 0.95)
  step 2  --RA-->  step 1   (conf 0.93)

pairs scored: 6   below threshold: 4


## Question 3 — how slow is all-pairs vs neighbours-only?

`window=None` compares every step to every other (thorough, but grows with the
square of the trace length). `window=2` compares only neighbours (cheap, but
blind to long-range links).

We time both on traces of realistic length. The number that matters is
**pairs per second**: multiply it out to a full training run (hundreds of
steps, 8 attempts each, hundreds of traces) and you learn whether all-pairs is
affordable. Write the result into `docs/run_log.md`; it decides the
`ari.window` setting in the configs.

In [5]:
import time

# Stand-ins for real traces of different lengths. Replace with real BBH traces
# once the corpus exists; this is enough to get a timing signal now.
short = trace.steps                       # ~3 steps
medium = trace.steps * 3                   # ~9 steps
long = trace.steps * 6                     # ~18 steps
sample = [short, medium, long]

for window in (None, 2):
    t0 = time.time()
    pairs = sum(ari.identify(s, window=window).n_pairs_scored for s in sample)
    dt = time.time() - t0
    rate = pairs / max(dt, 1e-6)
    print(f"window={str(window):4}  ->  {pairs:4d} pairs in {dt:5.2f}s   ({rate:6.1f} pairs/sec)")

print("\nRough projection for one training run:")
print("  ~400 steps x 8 attempts x ~250 traces = 800,000 traces")
print("  at, say, 10 steps each and all-pairs = ~45 pairs/trace = ~36M pairs")
print("  divide 36,000,000 by the pairs/sec above to get seconds of mining.")

window=None  ->   348 pairs in  0.58s   ( 601.8 pairs/sec)
window=2     ->    37 pairs in  0.08s   ( 464.3 pairs/sec)

Rough projection for one training run:
  ~400 steps x 8 attempts x ~250 traces = 800,000 traces
  at, say, 10 steps each and all-pairs = ~45 pairs/trace = ~36M pairs
  divide 36,000,000 by the pairs/sec above to get seconds of mining.


## What to record

Before closing this notebook, write into `docs/run_log.md`:

- the commit reference from the setup cell
- how many relation classes the model has (Q1)
- which way the arrows point (Q2)
- pairs per second for each window setting, and the resulting mining estimate (Q3)

Those four facts unblock the rest of the project. If any surprised you, that is
worth a short note in `docs/learning/` in your own words.